In [1]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency
import scipy.stats as stats
import numpy as np
import warnings

# Menyembunyikan peringatan
warnings.filterwarnings('ignore')

## 2x2 (Sex vs Target)

In [3]:
df = pd.read_csv('heart.csv')

df['sex_label'] = df['sex'].map({1: 'Male', 0: 'Female'})
df['target_label'] = df['target'].map({1: 'Disease', 0: 'No Disease'})

contingency_table_2d = pd.crosstab(df['sex_label'], df['target_label'])

print(contingency_table_2d)

target_label  Disease  No Disease
sex_label                        
Female            226          86
Male              300         413


#### Uji Independensi antar Variabel

In [8]:
chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table_2d)
print(f"Chi-Square: {chi2_stat:.4f}, p-value: {p_val:.4e}")
if p_val < 0.05:
    print("Kesimpulan: Tolak H0, terdapat hubungan (dependen) yang signifikan antara Jenis Kelamin dan Penyakit Jantung.\n")
else:
    print("Kesimpulan: Gagal tolak H0, Jenis Kelamin dan Penyakit Jantung saling independen.\n")

Chi-Square: 78.8631, p-value: 6.6568e-19
Kesimpulan: Tolak H0, terdapat hubungan (dependen) yang signifikan antara Jenis Kelamin dan Penyakit Jantung.



#### Model Loglinier Hierarkis & Uji Goodness of Fit

In [9]:
df_long1 = df.groupby(['sex_label', 'target_label']).size().reset_index(name='count')

model_2way_2d = smf.glm("count ~ sex_label * target_label", family=sm.families.Poisson(), data=df_long1).fit()

def goodness_of_fit(model, model_name):
    dev = model.deviance
    df_res = model.df_resid
    p_val = 1 - stats.chi2.cdf(dev, df_res) if df_res > 0 else 1.0 
    
    print(f"GoF {model_name}:")
    print(f"Deviance = {dev:.4f}, df = {df_res}, p-value = {p_val:.4f}")
    if df_res == 0:
         print("Kesimpulan: Model Saturated (Fit Sempurna terhadap data).\n")
    elif p_val > 0.05:
        print("Kesimpulan: Model FIT dengan data (Layak digunakan).\n")
    else:
        print("Kesimpulan: Model TIDAK FIT dengan data (Deviance terlalu besar).\n")

goodness_of_fit(model_2way_2d, "Model 2-Way (Interaksi 2 Arah)")

print(model_2way_2d.summary())

GoF Model 2-Way (Interaksi 2 Arah):
Deviance = 0.0000, df = 0, p-value = 1.0000
Kesimpulan: Model Saturated (Fit Sempurna terhadap data).

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  count   No. Observations:                    4
Model:                            GLM   Df Residuals:                        0
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -14.479
Date:                Wed, 08 Apr 2026   Deviance:                   1.6431e-14
Time:                        22:12:58   Pearson chi2:                 3.48e-26
No. Iterations:                     5   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                                                   coef    std err     

### **Interpretasi Section 1**
Berdasarkan hasil analisis:
1. **Uji Independensi**: Nilai p-value jauh di bawah 0.05, sehingga kita menolak H0. Terdapat hubungan signifikan antara jenis kelamin dan risiko penyakit jantung.
2. **Proporsi**: Dalam dataset ini, perempuan (Female) memiliki proporsi penyakit jantung yang lebih tinggi (~72%) dibandingkan laki-laki (Male, ~42%).
3. **Model Loglinier**: Koefisien interaksi `sex_label[T.Male]:target_label[T.No Disease]` bernilai positif (1.28), menunjukkan bahwa laki-laki dalam dataset ini cenderung memiliki frekuensi "No Disease" yang lebih tinggi secara signifikan dibandingkan pola pada perempuan.

## 2x2x2 (Sex vs Exang vs Target)

In [10]:
df['exang_label'] = df['exang'].map({1: 'Yes', 0: 'No'})

contingency_table_3d = pd.crosstab(df['sex_label'], [df['exang_label'], df['target_label']])

print(contingency_table_3d)

exang_label       No                Yes           
target_label Disease No Disease Disease No Disease
sex_label                                         
Female           202         36      24         50
Male             253        189      47        224


#### Uji Independensi Pairwise

In [11]:
def test_indep(v1, v2):
    ct = pd.crosstab(df[v1], df[v2])
    chi2, p, dof, ex = chi2_contingency(ct)
    print(f"{v1} vs {v2}, Chi-Square: {chi2:.4f}, p-value: {p:.4e}")
    if p < 0.05:
        print(f"Signifikan: Terdapat hubungan antara {v1} dan {v2}.")
    else:
        print(f"Tidak Signifikan: {v1} dan {v2} saling independen.")
    print()

test_indep('sex_label', 'target_label')
test_indep('sex_label', 'exang_label')
test_indep('exang_label', 'target_label')

sex_label vs target_label, Chi-Square: 78.8631, p-value: 6.6568e-19
Signifikan: Terdapat hubungan antara sex_label and target_label.

sex_label vs exang_label, Chi-Square: 19.2139, p-value: 1.1686e-05
Signifikan: Terdapat hubungan antara sex_label and exang_label.

exang_label vs target_label, Chi-Square: 194.8155, p-value: 2.8266e-44
Signifikan: Terdapat hubungan antara exang_label and target_label.



#### b. Model Loglinier Hierarkis & Uji Goodness of Fit

In [12]:
df_long2 = df.groupby(['sex_label', 'exang_label', 'target_label']).size().reset_index(name='count')

model_1way = smf.glm("count ~ sex_label + exang_label + target_label", 
                     family=sm.families.Poisson(), data=df_long2).fit()

model_2way = smf.glm("count ~ sex_label + exang_label + target_label + "
                     "sex_label:exang_label + sex_label:target_label + "
                     "exang_label:target_label", 
                     family=sm.families.Poisson(), data=df_long2).fit()

model_3way = smf.glm("count ~ sex_label * exang_label * target_label", 
                     family=sm.families.Poisson(), data=df_long2).fit()

goodness_of_fit(model_1way, "Model 1-Way (Main Effects)")
goodness_of_fit(model_2way, "Model 2-Way (Interaksi 2 Arah)")
goodness_of_fit(model_3way, "Model 3-Way (Saturated)")

print(model_3way.summary())

GoF Model 1-Way (Main Effects):
Deviance = 291.8614, df = 4, p-value = 0.0000
Kesimpulan: Model TIDAK FIT dengan data (Deviance terlalu besar).

GoF Model 2-Way (Interaksi 2 Arah):
Deviance = 2.8838, df = 1, p-value = 0.0895
Kesimpulan: Model FIT dengan data (Layak digunakan).

GoF Model 3-Way (Saturated):
Deviance = -0.0000, df = 0, p-value = 1.0000
Kesimpulan: Model Saturated (Fit Sempurna terhadap data).

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  count   No. Observations:                    8
Model:                            GLM   Df Residuals:                        0
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -25.372
Date:                Wed, 08 Apr 2026   Deviance:                  -2.6867e-14
Time:                        22:13:

#### Uji K-Ways dan K-Ways & Higher

In [13]:
dev_diff_2_higher = model_1way.deviance - model_3way.deviance
df_diff_2_higher = model_1way.df_resid - model_3way.df_resid
p_val_2_higher = 1 - stats.chi2.cdf(dev_diff_2_higher, df_diff_2_higher)

print("Uji 2-Ways & Higher Effect:")
print(f"G^2 = {dev_diff_2_higher:.4f}, df = {df_diff_2_higher}, p-value = {p_val_2_higher:.4e}")
if p_val_2_higher < 0.05:
    print("Kesimpulan: Terdapat efek interaksi 2-arah atau lebih yang signifikan.\n")

dev_diff_3 = model_2way.deviance - model_3way.deviance
df_diff_3 = model_2way.df_resid - model_3way.df_resid
p_val_3 = 1 - stats.chi2.cdf(dev_diff_3, df_diff_3)

print("Uji 3-Way Effect (Apakah butuh Saturated Model?):")
print(f"G^2 = {dev_diff_3:.4f}, df = {df_diff_3}, p-value = {p_val_3:.4f}")
if p_val_3 < 0.05:
    print("Kesimpulan: Interaksi 3-arah signifikan, gunakan Model Saturated.")
else:
    print("Kesimpulan: Interaksi 3-arah TIDAK signifikan, Model 2-Way sudah cukup untuk menjelaskan data.")

Uji 2-Ways & Higher Effect:
G^2 = 291.8614, df = 4, p-value = 0.0000e+00
Kesimpulan: Terdapat efek interaksi 2-arah atau lebih yang signifikan.

Uji 3-Way Effect (Apakah butuh Saturated Model?):
G^2 = 2.8838, df = 1, p-value = 0.0895
Kesimpulan: Interaksi 3-arah TIDAK signifikan, Model 2-Way sudah cukup untuk menjelaskan data.


## **Interpretasi Akhir**
Analisis loglinier pada tabel 2x2x2 (Sex, Exang, dan Target) menunjukkan:
1. **Model Terbaik**: Model 2-Way Interaction (Interaksi 2 Arah) memiliki nilai p-value Goodness of Fit sebesar **0.0894**, yang berarti model ini **fit** dengan data pada taraf signifikansi 5%.
2. **Interaksi 3-Arah**: Uji K-way = 3 menunjukkan p-value sebesar **0.0895**, sehingga interaksi tiga arah **tidak signifikan**. Model 2-way sudah cukup untuk menjelaskan hubungan antar variabel.
3. **Hubungan Utama**:
   - Terdapat hubungan kuat antara **Exang** (Exercise Induced Angina) dan **Target**. Pasien dengan Exang cenderung memiliki frekuensi "No Disease" yang lebih tinggi (koefisien interaksi ~2.03).
   - Jenis Kelamin juga berinteraksi dengan Target secara mandiri, namun tidak ada bukti kuat bahwa efek Exang terhadap Target berbeda secara signifikan antara laki-laki dan perempuan (karena interaksi 3-arah tidak signifikan).